In [1]:
import sumolib
import pandas as pd
from pathlib import Path

In [2]:
project_folder = Path.cwd()

if project_folder.name == "notebooks":
    project_folder = project_folder.parent

network_file = project_folder / "data" / "sumo" / "selected_area.net.xml"

net = sumolib.net.readNet(str(network_file))

print("SUMO network loaded successfully!")

print("Total Junctions:", len(net.getNodes()))
print("Total Roads:", len(net.getEdges(withInternal=False)))

SUMO network loaded successfully!
Total Junctions: 276
Total Roads: 794


In [3]:
junctions = []

for node in net.getNodes():
    x, y = node.getCoord()

    junctions.append({
        "id": node.getID(),
        "name": node.getID(),
        "x": x,
        "y": y
    })

junctions_df = pd.DataFrame(junctions)

print("Total Junctions:", len(junctions_df))

junctions_df.head()

Total Junctions: 276


,id,name,x,y
0,11158668350,11158668350,728.03,1407.83
1,8642758204,8642758204,697.99,1417.77
2,11586733428,11586733428,543.19,1069.27
3,cluster_11586733427_317194506,cluster_11586733427_317194506,507.93,1064.36
4,11586733429,11586733429,580.28,1069.12


In [4]:
roads = []

for edge in net.getEdges(withInternal=False):

    roads.append({
        "id": edge.getID(),
        "roadName": edge.getID(),
        "fromJunction": edge.getFromNode().getID(),
        "toJunction": edge.getToNode().getID(),
        "length": edge.getLength() / 1000,
        "speedLimit": edge.getSpeed() * 3.6,
        "lanes": edge.getLaneNumber(),
        "status": "Open"
    })

roads_df = pd.DataFrame(roads)

print("Total Roads:", len(roads_df))

roads_df.head()

Total Roads: 794


,id,roadName,fromJunction,toJunction,length,speedLimit,lanes,status
0,-1203892148,-1203892148,11158668350,8642758204,0.02439,50.004,1,Open
1,-1246284225,-1246284225,11586733428,cluster_11586733427_317194506,0.01955,50.004,1,Open
2,-1246284226#0,-1246284226#0,11586733428,11586733429,0.02703,50.004,1,Open
3,-1246284226#1,-1246284226#1,11586733440,11586733428,0.02518,50.004,1,Open
4,-1246285590,-1246285590,11586743529,11586743528,0.04177,50.004,1,Open


In [5]:
output_folder = project_folder / "data" / "outputs"

output_folder.mkdir(parents=True, exist_ok=True)

junctions_df.to_csv(
    output_folder / "junctions.csv",
    index=False
)

roads_df.to_csv(
    output_folder / "roads.csv",
    index=False
)

print("CSV files saved successfully!")

CSV files saved successfully!


In [6]:
print("Junctions:", len(junctions_df))
print("Roads:", len(roads_df))

print("\nMissing Values:")
print(roads_df.isnull().sum())

print("\nSample Roads:")
print(roads_df.head())

Junctions: 276
Roads: 794

Missing Values:
id              0
roadName        0
fromJunction    0
toJunction      0
length          0
speedLimit      0
lanes           0
status          0
dtype: int64

Sample Roads:
              id       roadName fromJunction                     toJunction  \
0    -1203892148    -1203892148  11158668350                     8642758204   
1    -1246284225    -1246284225  11586733428  cluster_11586733427_317194506   
2  -1246284226#0  -1246284226#0  11586733428                    11586733429   
3  -1246284226#1  -1246284226#1  11586733440                    11586733428   
4    -1246285590    -1246285590  11586743529                    11586743528   

    length  speedLimit  lanes status  
0  0.02439      50.004      1   Open  
1  0.01955      50.004      1   Open  
2  0.02703      50.004      1   Open  
3  0.02518      50.004      1   Open  
4  0.04177      50.004      1   Open  


In [7]:
junction_ids = set(junctions_df["id"])

valid = all(
    a in junction_ids and b in junction_ids
    for a, b in zip(
        roads_df["fromJunction"],
        roads_df["toJunction"]
    )
)

print("All roads connect to valid junctions:", valid)

All roads connect to valid junctions: True
